# Finetuning Using Unsloth SFTTrainer
This notebook finetunes the language, attention and mlp modules of Unsloth's Qwen3-VL-2B-Instruct-unsloth-bnb-4bit model using the created EK55-MCQ dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/DL25_Project/EPIC-KITCHENS55-SAMPLED-FRAMES/EPIC-KITCHENS55-SAMPLED-FRAMES.zip /content
!unzip -q /content/EPIC-KITCHENS55-SAMPLED-FRAMES.zip -d /content/

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastVisionModel
import torch
import pandas as pd

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.41G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None
)

<a name="Data"></a>
### Data Prep

In [ ]:
import json
import random
import os
from PIL import Image
from tqdm import tqdm

# Configuration
DATASET_PATH = "/content/avion_distractors_combined.jsonl"
FRAMES_ROOT = "/content/content/EPIC-KITCHENS55-SAMPLED-FRAMES"
SPLIT_CSV_PATH = "/content/ek55_data_split.csv"
N_FRAMES = 16


In [ ]:
def construct_mcq_prompt(ground_truth, distractors):
    options = [ground_truth['narration']] + [d['answer'] for d in distractors[:4]]
    random.shuffle(options)

    labels = ['A', 'B', 'C', 'D', 'E'][:len(options)]
    option_text_lines = []
    correct_label = None

    for label, text in zip(labels, options):
        option_text_lines.append(f"{label}. {text}")
        if text == ground_truth['narration']:
            correct_label = label

    options_block = "\n".join(option_text_lines)
    prompt_text = (
        "Analyze the video frames and select the correct action from the given options.\n"
        "Answer with ONLY the option letter (A, B, C, D, or E). Do not explain.\n\n"
        f"{options_block}"
    )

    return prompt_text, correct_label

def get_image_paths(entry):
    pid = entry['participant_id']
    vid = entry['video_id']
    frame_indices = entry['frame_indices']

    # Subsample 16 frames uniformly
    indices = [frame_indices[i * len(frame_indices) // N_FRAMES] for i in range(N_FRAMES)]


    images = []
    for idx in indices:
        # Construct path matching the subsample_dataset.py logic
        # Structure: root/pid/rgb/vid/frame_0000000123.jpg
        path = os.path.join(FRAMES_ROOT, pid, 'rgb', vid, f"frame_{idx:010d}.jpg")
        #img = Image.open(path).convert("RGB").resize((224, 224))
        img = Image.open(path).convert("RGB")
        images.append(img)

    return images


In [ ]:
def convert_to_conversation(entry):
    prompt, answer = construct_mcq_prompt(entry['ground_truth'], entry['distractors_with_confidence'])
    images = get_image_paths(entry)

    # Construct content
    user_content = []
    for img in images:
        user_content.append({"type": "image", "image": img})
    user_content.append({"type": "text", "text": prompt})

    conversation = [
        { "role": "user",
          "content" : user_content
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : answer} ]
        },
    ]
    return { "messages" : conversation }


Convert the dataset into the correct format for finetuning:

In [ ]:
# Load JSONL Data
raw_data = []
with open(DATASET_PATH, 'r') as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line))


# Load Splits from CSV
split_df = pd.read_csv(SPLIT_CSV_PATH)
video_to_split = dict(zip(split_df['video_id'], split_df['split']))

# Convert and Split
train_dataset = []
eval_dataset = []
test_excluded_count = 0

for entry in tqdm(raw_data):
    vid = entry['video_id']
    split = video_to_split.get(vid, 'train')

    if split == 'test':
        test_excluded_count += 1
        continue

    conv = convert_to_conversation(entry)

    if split == 'eval':
        eval_dataset.append(conv)
    else:
        train_dataset.append(conv)

print(f"Training Samples: {len(train_dataset)}")
print(f"Evaluation Samples: {len(eval_dataset)}")
print(f"Test Samples: {test_excluded_count}")

print("Example User Prompt:\n", train_dataset[0]['messages'][0]['content'][-1]['text'])


100%|██████████| 6342/6342 [01:23<00:00, 76.39it/s]

Training Samples: 4529
Evaluation Samples: 573
Test Samples: 1240
Example User Prompt (Train):
 Analyze the video frames and select the correct action from the given options.
Answer with ONLY the option letter (A, B, C, D, or E). Do not explain.

A. search in cupboard
B. open light
C. open ventilator
D. search for something
E. open door


In [ ]:
train_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=456x256>},
    {

<a name="Train"></a>
### Train the model


In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 30,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "qwen3_2b_finetune",
        report_to = "wandb",     # For Weights and Biases

        # Evaluation & Saving Config
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 50,
        metric_for_best_model = "eval_loss",
        load_best_model_at_end = True,
        per_device_eval_batch_size = 8,

        # For vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)


Unsloth: Model does not have a default image size - using 512


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.318 GB.
20.16 GB of memory reserved.


In [ ]:
import wandb
wandb.init(project="huggingface", id="ixu2jrn9", resume="must")

In [ ]:
# trainer_stats = trainer.train()
trainer_stats = trainer.train(resume_from_checkpoint=True)

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,529 | Num Epochs = 10 | Total steps = 1,420
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 17,432,576 of 2,144,964,608 (0.81% trained)


Step,Training Loss,Validation Loss
900,0.228000,0.372008
950,0.241100,0.359310
1000,0.180400,0.369672
1050,0.188800,0.371329
1100,0.229100,0.365098


Step,Training Loss,Validation Loss
900,0.228000,0.372008
950,0.241100,0.359310
1000,0.180400,0.369672
1050,0.188800,0.371329
1100,0.229100,0.365098
1150,0.179500,0.373423
1200,0.176200,0.383990


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("/content/drive/MyDrive/qwen2_2b_without_vision_v4")  # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/qwen2_2b_without_vision_v4")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

In [ ]:
!cp -r /content/qwen3_2b_finetune /content/drive/MyDrive/